# Run Jasna in Google Colab (free tier)

This notebook runs the official Jasna Linux NVIDIA release with the **free BasicVSR++ model**. It does not require Docker, a license code, or a paid GPU service.

> **Free Colab GPUs are not guaranteed.** Choose **Runtime → Change runtime type → T4 GPU** if it is offered. Jasna requires an NVIDIA GPU with compute capability 7.5+ and Linux driver 580+. Colab may assign unsupported hardware, run out of free GPU quota, or have an older host driver; the checks below will stop with an explanation in that case. This notebook cannot upgrade Colab's host driver or bypass Jasna's checks.

Keep Jasna installed under `/content` (the VM's local disk), not Google Drive. This downloads several GB, and the first model/engine setup takes time. Jasna's cloud guide recommends at least 80 GB of local disk; a free Colab runtime may not have enough space, and the notebook cannot increase its disk. The Colab VM is temporary: download/copy your result to Drive before the session ends. For privacy, run this notebook only in your own Colab account and do not share a notebook containing private media paths.

## 1. Check the free GPU and driver

Select a GPU runtime first (`Runtime → Change runtime type → T4 GPU`, if available), then run this cell. If Colab gives you a CPU runtime or a driver below Jasna's minimum, stop here; a free session may work later when a compatible GPU is available.

In [ ]:
#@title Check the free GPU and driver
import re, subprocess, torch

if not torch.cuda.is_available():
    raise RuntimeError('No NVIDIA GPU is attached. In Colab, choose Runtime → Change runtime type → a GPU, then reconnect.')
gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
driver = subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).splitlines()[0].strip()
driver_tuple = tuple(map(int, re.findall(r'\d+', driver)[:3]))
print(f'GPU: {gpu_name} | compute capability: {capability[0]}.{capability[1]} | driver: {driver}')
if capability < (7, 5):
    raise RuntimeError('Jasna requires compute capability 7.5 or newer; this Colab GPU is too old.')
if driver_tuple < (580,):
    raise RuntimeError(f'Jasna requires Linux NVIDIA driver 580+; this Colab runtime has {driver}. Colab controls the host driver, so try a later compatible runtime or another GPU service.')
print('Basic hardware checks passed. Jasna performs its own additional compatibility checks when it starts.')

## 2. Install Jasna locally in this Colab session

This fetches the official `v0.12.3` Linux NVIDIA release parts from GitHub, unpacks them to `/content`, and installs the small system libraries Jasna needs. The archive is multi-gigabyte; keep this cell running until it finishes.

In [ ]:
#@title Install the official Jasna runtime
import json, pathlib, re, shlex, shutil, subprocess, urllib.request

JASNA_TAG = 'v0.12.3'  # Pinned official release; change only to a published Linux NVIDIA release.
root = pathlib.Path('/content')
install_root = root / 'jasna-install'
install_root.mkdir(exist_ok=True)
api_url = f'https://api.github.com/repos/Kruk2/jasna/releases/tags/{JASNA_TAG}'
with urllib.request.urlopen(api_url) as response:
    release = json.load(response)
version = release['tag_name'].removeprefix('v')
part_pattern = re.compile(r'^jasna-linux-nvidia-' + re.escape(version) + r'\.tar\.zst\.part\d+$')
parts = sorted((item for item in release.get('assets', []) if part_pattern.fullmatch(item['name'])), key=lambda item: item['name'])
if not parts:
    raise RuntimeError(f'No official Linux NVIDIA archive parts found for {JASNA_TAG}.')

subprocess.run(['apt-get', 'update', '-qq'], check=True)
for package in ('zstd', 'libgl1', 'libglib2.0-0'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', package], check=True)

archive_dir = root / 'jasna-release-parts'
archive_dir.mkdir(exist_ok=True)
for item in parts:
    destination = archive_dir / item['name']
    if not destination.exists() or destination.stat().st_size != item['size']:
        print(f'Downloading {item["name"]} ({item["size"] / 1e9:.1f} GB) ...')
        subprocess.run(['wget', '--show-progress', '--progress=dot:giga', '-O', str(destination), item['browser_download_url']], check=True)

extraction_dir = install_root / 'unpacked'
extraction_dir.mkdir(exist_ok=True)
binary_dir = install_root / f'jasna-linux-nvidia-{version}'
if not (binary_dir / 'jasna').is_file():
    quoted_parts = ' '.join(shlex.quote(str(archive_dir / item['name'])) for item in parts)
    command = f'cat {quoted_parts} | zstd -dc | tar -xpf - --same-owner -C {shlex.quote(str(extraction_dir))}'
    subprocess.run(['bash', '-o', 'pipefail', '-c', command], check=True)
    extracted = extraction_dir / f'jasna-linux-nvidia-{version}'
    if not (extracted / 'jasna').is_file():
        raise RuntimeError(f'Expected Jasna executable not found in {extracted}.')
    if binary_dir.exists():
        shutil.rmtree(binary_dir)
    extracted.rename(binary_dir)
jasna = binary_dir / 'jasna'
print(f'Jasna {JASNA_TAG} is ready at {jasna}')

## 3. Automatically find every video in Drive

The notebook recursively scans `My Drive/jasna` for `.mp4`, `.mkv`, `.avi`, `.mov`, `.wmv`, `.flv`, and `.webm` files, copies them onto the Colab VM, and saves results under `My Drive/jasna/Restored/`. It skips that output folder and does not change your originals.

The code cells are collapsed by default; their output and progress remain visible. Use **Show code** on a cell if you want to inspect or edit it.

In [ ]:
#@title Find videos in Drive /jasna
from google.colab import drive
import pathlib, shutil

drive.mount('/content/drive')
drive_folder = pathlib.Path('/content/drive/MyDrive/jasna')
restored_drive_folder = drive_folder / 'Restored'
video_extensions = {'.mp4', '.mkv', '.avi', '.mov', '.wmv', '.flv', '.webm'}

if not drive_folder.is_dir():
    raise FileNotFoundError(f'Expected your Drive folder at {drive_folder}')
video_files = sorted(
    (p for p in drive_folder.rglob('*') if p.is_file() and restored_drive_folder not in p.parents and p.suffix.lower() in video_extensions),
    key=lambda p: p.as_posix().casefold(),
)
if not video_files:
    raise FileNotFoundError(f'No supported video files found inside {drive_folder}')

local_input_dir = pathlib.Path('/content/jasna-batch-input')
local_output_dir = pathlib.Path('/content/jasna-batch-output')
shutil.rmtree(local_input_dir, ignore_errors=True)
shutil.rmtree(local_output_dir, ignore_errors=True)
local_input_dir.mkdir(parents=True)
local_output_dir.mkdir(parents=True)
video_manifest = []
for index, source_video in enumerate(video_files, start=1):
    # Use an ASCII-only local name and unique index to avoid path/filename issues and collisions.
    local_video = local_input_dir / f'video_{index:04d}{source_video.suffix.lower()}'
    print(f'Copying {source_video.name} ({source_video.stat().st_size / 1e9:.2f} GB)')
    shutil.copy2(source_video, local_video)
    video_manifest.append((local_video, source_video))

print(f'Queued {len(video_manifest)} video(s). Outputs will go in {restored_drive_folder}.')

## 4. Restore with the free model

This runs free BasicVSR++ without a supporter license. Colab's GPU allotment can end or disconnect during long jobs; keep a backup of the original video.

In [ ]:
#@title Restore all videos and save results to Drive
import os, subprocess, shutil

# Colab form controls: default detector settings are recommended; tune after checking a result.
USE_LARGE_DETECTOR = False #@param {type:"boolean"}
DETECTION_THRESHOLD = "model default" #@param ["model default", "0.30", "0.25", "0.40", "0.45"]
OUTPUT_CQ = 20 #@param {type:"slider", min:15, max:35, step:1}

detector = 'rfdetr-v6'
if USE_LARGE_DETECTOR:
    # Jasna's optional larger detector can help on 4K video, but is slower and downloads ~135 MB.
    detector = 'rfdetr-v6-large'
    model_path = binary_dir / 'model_weights' / 'rfdetr-v6-large.onnx'
    model_path.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'wget', '-c', '--show-progress', '--progress=dot:giga', '-O', str(model_path),
        'https://github.com/Kruk2/jasna/releases/download/0.1/rfdetr-v6-large.onnx',
    ], check=True)

env = os.environ.copy()
env['XDG_CONFIG_HOME'] = '/content/jasna-config'  # BasicVSR++ is free; no license code needed.
cmd = [
    str(jasna),
    '--input', str(local_input_dir),
    '--output', str(local_output_dir),
    '--output-pattern', '{original}_restored.mkv',
    '--restoration-model-name', 'basicvsrpp',
    '--detection-model', detector,
    '--codec', 'hevc',
    '--cq', str(OUTPUT_CQ),
]
if DETECTION_THRESHOLD != 'model default':
    cmd += ['--detection-score-threshold', DETECTION_THRESHOLD]

result = subprocess.run(cmd, cwd=str(binary_dir), env=env, check=False)
restored_drive_folder.mkdir(parents=True, exist_ok=True)
restored_count = 0
for index, (local_video, drive_video) in enumerate(video_manifest, start=1):
    local_result = local_output_dir / f'{local_video.stem}_restored.mkv'
    if not local_result.is_file():
        print(f'No output found for {drive_video.name}')
        continue
    drive_result = restored_drive_folder / f'{index:04d}_{drive_video.stem}_restored.mkv'
    shutil.copy2(local_result, drive_result)
    print(f'Saved: {drive_result}')
    restored_count += 1
print(f'Finished: {restored_count}/{len(video_manifest)} video(s) copied to Drive.')
if result.returncode:
    raise RuntimeError(f'Jasna returned exit code {result.returncode}; see the visible output above.')

## Improving results (still using free models)

- BasicVSR++ remains the free restoration model. For better mosaic **detection**, turn on **USE_LARGE_DETECTOR** for 4K footage; it is slower and downloads an extra model. The built-in `rfdetr-v6` is a good fast default.
- If mosaics are missed, try detection threshold `0.30` (lower finds more, but may also restore clean areas). If the result changes non-mosaic areas, return to `model default` or raise the threshold. Inspect a short sample before running a long batch.
- Output HEVC CQ defaults to `20` here; lower CQ means less encoding loss and larger files. This improves output encoding fidelity, not the AI restoration itself.
- A more detailed restoration model (LTX) is a paid supporter model, so this free notebook does not enable it. No setting can guarantee perfect mosaic removal; results depend on source quality and detection.

## Free-tier limitations

- Free Colab GPU access, GPU type, driver version, disk space, and session length are controlled by Google and may change. There is no way for this project to guarantee a free GPU or make an incompatible runtime work.
- Jasna's model preparation can take a long time, and a disconnected Colab session loses files under `/content`; copy results to Drive before the session expires.
- Use Jasna's official documentation for its supported GPU/driver list and the latest release details: https://github.com/Kruk2/jasna/blob/main/docs/en/cloud.md